# Student Performance Dashboard
## Milestone 1: Data Collection & Preprocessing
Steps: load → inspect → clean → categorize → save cleaned CSV (ready for SQL).

## 1. Load the data

In [1]:
import pandas as pd

# Read everything as text first, so messy values (like '53.3%' or '61,7') don't break loading
df = pd.read_csv('students_performance.csv', dtype=str)
print('Shape:', df.shape)
df.head(10)

Shape: (10350, 6)


,student_id,name,subject,score,date,attendance
0,S2023610,Malak Ahmed Ibrahim,Mathematics,45.9,2025-12-21,85.5
1,S2024867,KAREEM ABDELAZIZ GAMAL,Database System,57.1,2026-04-08,82.8
2,S2023798,Farida Zaki Badawy,statistics,74.4,01 Apr 2026,92.9
3,2024259,Rana Mahmoud Rizk,Economics,24.9,2025-10-21,71
4,S2024913,Ramy Metwally Ibrahim,Data Warehouse,52.7,2026-04-02,85.9
5,S2023144,Ibrahim Gamal Badawy,Chemistry,71.2,2025-10-16,100
6,S2024057,Menna Mansour Fathy,Arabic,NaN,2026/02/09,0.742
7,S2023450,Marwan Eid Abdelaziz,Statistics,66.5,20 Dec 2025,89.6
8,S2024453,Lamis Hassan Taha,Chemistry,60.8,2025-12-14,91.1
9,s2024370,Nermine Mostafa Magdy,Arabic,71.8,2026-04-04,92.2


## 2. Inspect the problems

In [2]:
print('Missing values:')
print(df.isna().sum())
print('\nExact duplicate rows:', df.duplicated().sum())
print('\nUnique subject spellings:', df['subject'].nunique())

Missing values:
student_id     40
name           87
subject        68
score         266
date           81
attendance    281
dtype: int64

Exact duplicate rows: 250

Unique subject spellings: 53


## 3. Clean the data
Keep a copy of the raw data, then fix one column at a time.

In [3]:
raw_rows = len(df)
data = df.copy()

# Standardize placeholder values as real nulls
data = data.replace(r'^\s*(-|\?|na|n/a|nan|none|absent|ab|)\s*$', pd.NA, regex=True)

### 3.1 student_id
Fix spacing, lowercase 's', and the missing 'S' prefix.

In [4]:
data['student_id'] = data['student_id'].str.strip().str.upper()
data['student_id'] = data['student_id'].str.replace(r'^(\d)', r'S\1', regex=True)
data = data.dropna(subset=['student_id'])   # a row without an ID is useless

### 3.2 name
Fix spacing/capitalization. If a student's name is missing, fill it from their other rows.

In [5]:
data['name'] = data['name'].str.strip().str.replace(r'\s+', ' ', regex=True).str.title()

# most common name per student_id
name_map = data.dropna(subset=['name']).groupby('student_id')['name'].agg(lambda s: s.mode()[0])
data['name'] = data['student_id'].map(name_map)   # one consistent name per ID
data = data.dropna(subset=['name'])

### 3.3 subject
Many typos / abbreviations → map them to one standard name.

In [6]:
subject_map = {
    'mathematics': 'Mathematics', 'math': 'Mathematics', 'maths': 'Mathematics', 'mathmatics': 'Mathematics',
    'statistics': 'Statistics', 'stats': 'Statistics', 'statistcs': 'Statistics',
    'physics': 'Physics', 'phys': 'Physics', 'physcis': 'Physics',
    'chemistry': 'Chemistry', 'chemestry': 'Chemistry', 'chem': 'Chemistry',
    'biology': 'Biology', 'bio': 'Biology', 'biolgy': 'Biology',
    'english': 'English', 'eng': 'English',
    'arabic': 'Arabic',
    'history': 'History', 'histroy': 'History',
    'economics': 'Economics', 'econ': 'Economics', 'economcs': 'Economics',
    'computer science': 'Computer Science', 'comp sci': 'Computer Science',
    'cs': 'Computer Science', 'computer sceince': 'Computer Science',
    'database systems': 'Database Systems', 'database system': 'Database Systems',
    'db systems': 'Database Systems', 'databases': 'Database Systems',
    'data warehousing': 'Data Warehousing', 'data warehouse': 'Data Warehousing', 'dwh': 'Data Warehousing',
}
data['subject'] = data['subject'].str.strip().str.lower().map(subject_map)
data = data.dropna(subset=['subject'])
print(sorted(data['subject'].unique()))

['Arabic', 'Biology', 'Chemistry', 'Computer Science', 'Data Warehousing', 'Database Systems', 'Economics', 'English', 'History', 'Mathematics', 'Physics', 'Statistics']


### 3.4 score
Remove `%`, change `,` to `.`, convert to number, and treat impossible values (<0 or >100) as missing.

In [7]:
data['score'] = (data['score'].str.replace('%', '', regex=False)
                              .str.replace(',', '.', regex=False)
                              .str.strip())
data['score'] = pd.to_numeric(data['score'], errors='coerce')
data.loc[~data['score'].between(0, 100), 'score'] = pd.NA

print('Missing scores:', data['score'].isna().sum())
data = data.dropna(subset=['score'])   # score is the main measure, so we don't invent it

Missing scores: 553


### 3.5 attendance
Same cleaning. Some values are fractions (0.742 = 74.2%), so convert them to percentages. Missing attendance is filled with the median.

In [8]:
data['attendance'] = (data['attendance'].str.replace('%', '', regex=False)
                                        .str.replace(',', '.', regex=False)
                                        .str.strip())
data['attendance'] = pd.to_numeric(data['attendance'], errors='coerce')

# fractions (0-1) -> percentage
data.loc[data['attendance'] <= 1, 'attendance'] *= 100
# impossible values -> null
data.loc[~data['attendance'].between(0, 100), 'attendance'] = pd.NA

# fix nulls with the median
data['attendance'] = data['attendance'].fillna(data['attendance'].median()).round(1)

### 3.6 date
The file mixes several formats (`2025-12-21`, `21/12/2025`, `01 Apr 2026`, `April 01, 2026`...). We try each format and combine.

In [9]:
formats = ['%Y-%m-%d', '%d/%m/%Y', '%d-%m-%Y', '%d.%m.%Y', '%Y/%m/%d', '%d %b %Y', '%B %d, %Y']

def parse_date(s):
    out = pd.Series(pd.NaT, index=s.index, dtype='datetime64[ns]')
    for f in formats:
        out = out.fillna(pd.to_datetime(s, format=f, errors='coerce'))
    return out

data['date'] = parse_date(data['date'].str.strip())
print('Unparsed dates:', data['date'].isna().sum())
data = data.dropna(subset=['date'])
data['date'] = data['date'].dt.date   # keep YYYY-MM-DD only

Unparsed dates: 105


### 3.7 Remove duplicates
Done last, because after cleaning, rows that looked different become identical.

In [10]:
before = len(data)
data = data.drop_duplicates()
# same student + subject + date should appear only once
data = data.drop_duplicates(subset=['student_id', 'subject', 'date'], keep='first')
print('Duplicates removed:', before - len(data))

Duplicates removed: 389


## 4. Categorize performance
- **High**: score ≥ 80
- **Medium**: 60 ≤ score < 80
- **Low**: score < 60

In [11]:
def categorize(score):
    if score >= 80:
        return 'High'
    elif score >= 60:
        return 'Medium'
    return 'Low'

data['performance'] = data['score'].apply(categorize)
data['performance'].value_counts()

performance
Medium    4976
High      2215
Low       1956
Name: count, dtype: int64

## 5. Final check

In [12]:
data = data.sort_values(['student_id', 'date']).reset_index(drop=True)

print(f'Rows: {raw_rows} -> {len(data)}')
print('Students:', data['student_id'].nunique())
print('Nulls left:', data.isna().sum().sum())
print()
data.info()
data.head(10)

Rows: 10350 -> 9147


Students: 1000
Nulls left: 0

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9147 entries, 0 to 9146
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   student_id   9147 non-null   object 
 1   name         9147 non-null   object 
 2   subject      9147 non-null   object 
 3   score        9147 non-null   float64
 4   date         9147 non-null   object 
 5   attendance   9147 non-null   float64
 6   performance  9147 non-null   object 
dtypes: float64(2), object(5)
memory usage: 500.4+ KB


,student_id,name,subject,score,date,attendance,performance
0,S2023002,Rania Nasser Morsi,Mathematics,52.4,2025-10-20,81.3,Low
1,S2023002,Rania Nasser Morsi,Computer Science,73.3,2025-12-14,81.1,Medium
2,S2023002,Rania Nasser Morsi,Mathematics,72.6,2025-12-15,96.2,Medium
3,S2023002,Rania Nasser Morsi,Database Systems,60.6,2025-12-22,92.5,Medium
4,S2023002,Rania Nasser Morsi,Database Systems,64.8,2026-02-19,82.5,Medium
5,S2023002,Rania Nasser Morsi,Data Warehousing,75.8,2026-02-19,84.6,Medium
6,S2023002,Rania Nasser Morsi,Data Warehousing,64.9,2026-02-22,90.9,Medium
7,S2023005,Dina Khalil Ali,Arabic,76.7,2025-10-18,84.0,Medium
8,S2023005,Dina Khalil Ali,Biology,65.9,2025-10-24,80.3,Medium
9,S2023005,Dina Khalil Ali,Physics,51.8,2025-12-18,71.6,Low


## 6. Save the cleaned dataset (ready for SQL import)

In [13]:
data.to_csv('students_performance_clean.csv', index=False)
print('Saved: students_performance_clean.csv')

Saved: students_performance_clean.csv
